# Phase 2 — Separate validation-only stabilization run

Run on the connected Kaggle GPU kernel, in order. This notebook embeds the source
and frozen pilot manifest. Initial runs use seed 42 and 16 base channels; they are
pilot baselines, not exact paper reproductions. Only one GPU is used deliberately.
The original Dice-only runs collapsed and are preserved. This v2 run adds
GroupNorm, BCE+Dice, gradient clipping and a lower learning rate. Holdout/query
scores are excluded from this diagnostic run. No improvement is claimed yet.

Recovery tests must pass before long training. Checkpoints are local to the live
session until you download/save outputs. Keep the session running until backup
is verified. No snapshots are automatically deleted.


In [1]:
import base64, hashlib, io, json, os, sys, zipfile
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
payload=base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='f06c6d29f616b3ebc5dfaab64b217f684d97ea3637bf8c110f297b3fbc3b7f44'
project=Path('/kaggle/working/road_crack_phase2_v2')
project.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    for name in z.namelist():
        path=(project/name).resolve()
        assert path.is_relative_to(project.resolve())
        if path.exists(): assert path.read_bytes()==z.read(name), 'Code differs: preserve this run and use a new project directory'
    z.extractall(project)
os.chdir(project)
manifest=project/'data/manifests/pilot_v1.json'
assert hashlib.sha256(manifest.read_bytes()).hexdigest()==manifest.with_suffix('.json.sha256').read_text().split()[0]
import subprocess
restoration_report=project/'artifacts/dataset_restore.json'
subprocess.run([sys.executable,'scripts/restore_datasets.py',
    '--manifest',str(manifest),'--sources','configs/data_sources.json',
    '--destination','/kaggle/working/road_crack_phase1/data/raw',
    '--report',str(restoration_report)],check=True)
data_root=Path(json.loads(restoration_report.read_text())['data_root'])
import torch
assert torch.cuda.is_available(), 'Enable Kaggle GPU'
print('GPU:',torch.cuda.get_device_name(0),'Devices:',torch.cuda.device_count(),'Torch:',torch.__version__)
print('Frozen splits:',json.loads(manifest.read_text())['counts'])


Reused existing data; all image/mask checksums passed: /kaggle/working/road_crack_phase1/data/raw
GPU: Tesla T4 Devices: 2 Torch: 2.11.0+cu128
Frozen splits: {'source_holdout': 237, 'source_train': 240, 'source_validation': 60, 'target_acquisition': 25, 'target_query': 95}


In [2]:
import subprocess, xml.etree.ElementTree as ET
evidence=project/'artifacts/phase2'
evidence.mkdir(parents=True,exist_ok=True)
test_run=subprocess.run([sys.executable,'-m','pytest','-q','-rA',
    '--junitxml='+str(evidence/'tests.xml')],text=True,capture_output=True)
(evidence/'tests.txt').write_text(test_run.stdout+test_run.stderr)
print(test_run.stdout,test_run.stderr)
assert test_run.returncode==0, 'Recovery tests failed; do not start training'
cases=ET.parse(evidence/'tests.xml').findall('.//testcase')
resume_cases=[c for c in cases if 'test_actual_trainer_mid_epoch_resume' in c.attrib.get('name','')]
assert len(resume_cases)==4 and all(c.find('skipped') is None for c in resume_cases), 'Both CPU and CUDA recovery tests must execute'
(evidence/'requirements.kaggle.lock.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
subprocess.run([sys.executable,'scripts/environment.py'],check=True)
print('Actual U-Net and R2AU trainer recovery tests passed on CPU and CUDA.')


...................                                                      [100%]
==================================== PASSES ====================================
________________ test_actual_trainer_mid_epoch_resume[cpu-unet] ________________
----------------------------- Captured stdout call -----------------------------
{"epoch": 1, "step": 2, "train_loss": 1.3289219737052917, "validation": {"loss": 0.8950339555740356, "dice": 0.0, "iou": 0.0, "precision": 0.0, "recall": 0.0, "predicted_foreground_fraction": 0.0, "truth_foreground_fraction": 0.0625, "n": 2}, "learning_rate": 0.001}
{"epoch": 2, "step": 4, "train_loss": 1.300324559211731, "validation": {"loss": 0.8954219222068787, "dice": 0.0, "iou": 0.0, "precision": 0.0, "recall": 0.0, "predicted_foreground_fraction": 0.0, "truth_foreground_fraction": 0.0625, "n": 2}, "learning_rate": 0.001}
{"epoch": 1, "step": 2, "train_loss": 1.3289219737052917, "validation": {"loss": 0.8950339555740356, "dice": 0.0, "iou": 0.0, "precision": 0.0, 

## Training

The following cell starts both pilot baselines (up to 100 epochs each, early
stopping patience 15). Training is FP32, deterministic, batch eight, 256 × 256.
Checkpoints are saved at each epoch and every 30 steps. Rerunning resumes an
existing checkpoint; configurations and dataset hashes must match exactly.

Interrupt requests handled by the training process stop at a complete optimizer
step. A forced kernel kill can lose progress since the latest saved checkpoint.
To stop from another cell/terminal, send SIGTERM to the printed worker PID; avoid
shutting down the kernel/session. If disk space runs low, export then resume on
adequate persistent storage; older snapshots are preserved.


In [3]:
import shutil
assert shutil.disk_usage('/kaggle/working').free>8*1024**3, 'Need at least 8 GiB free before the two pilot runs'
def execute_logged(arguments,log_path):
    with open(log_path,'a',buffering=1) as log:
        process=subprocess.Popen(arguments,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        print('Training worker PID:',process.pid,flush=True)
        try:
            for line in process.stdout:
                print(line,end='',flush=True); log.write(line)
            code=process.wait()
        except KeyboardInterrupt:
            import signal
            process.send_signal(signal.SIGTERM)
            process.wait(timeout=120)
            raise
    if code: raise RuntimeError('Worker failed; inspect '+str(log_path))
for architecture in ['unet','r2au']:
    run=project/'artifacts/phase2'/architecture
    command=[sys.executable,'scripts/train_baseline.py','--config','configs/stabilized_'+architecture+'.json',
             '--manifest',str(manifest),'--data-root',str(data_root),'--run-dir',str(run)]
    if (run/'checkpoints/latest.json').exists(): command.append('--resume')
    execute_logged(command,evidence/(architecture+'_console.log'))
    status=json.loads((run/'status.json').read_text())
    if status['state']!='training_complete':
        print('Stopped safely; export checkpoints before resuming.'); break
    print('Completed',architecture,json.loads((run/'evaluation.json').read_text()))


Training worker PID: 274
{"epoch": 1, "step": 30, "train_loss": 1.3269481182098388, "validation": {"loss": 0.9066696524620056, "dice": 0.5417952542891726, "iou": 0.42417633247484143, "precision": 0.5553918056810896, "recall": 0.6145111453486607, "predicted_foreground_fraction": 0.03579661051432292, "truth_foreground_fraction": 0.028463999430338543, "n": 60}, "learning_rate": 0.0001}
{"epoch": 2, "step": 60, "train_loss": 1.2037814537684122, "validation": {"loss": 0.8774870951970418, "dice": 0.6045834245781104, "iou": 0.47866438596198957, "precision": 0.5652057676265637, "recall": 0.7061801263441642, "predicted_foreground_fraction": 0.037484232584635416, "truth_foreground_fraction": 0.028463999430338543, "n": 60}, "learning_rate": 0.0001}
{"epoch": 3, "step": 90, "train_loss": 1.1614625056584675, "validation": {"loss": 0.863213316599528, "dice": 0.6209769253929456, "iou": 0.49379553149143857, "precision": 0.5344132127861182, "recall": 0.779911262790362, "predicted_foreground_fraction": 

In [4]:
# Profile each completed validation-best model on the same GPU and CPU.
for architecture in ['unet','r2au']:
    run=evidence/architecture
    if not (run/'evaluation.json').exists(): continue
    for device in ['cuda','cpu']:
        execute_logged([sys.executable,'scripts/profile_baseline.py','--run-dir',str(run),
                        '--device',device],evidence/(architecture+'_profile.log'))


Training worker PID: 292
{
  "architecture": "unet",
  "device": "cuda",
  "hardware": "Tesla T4",
  "threads": 4,
  "batch_size": 1,
  "image_size": 256,
  "parameters": 1965569,
  "p50_ms": 6.028796999999031,
  "p95_ms": 6.923550999999861,
  "warmups": 20,
  "timed_iterations": 200,
  "scope": "model pass only; no image decoding/transfer",
  "recovery_checkpoint_bytes": 23705833,
  "fp32_weight_bytes": 7862276,
  "cuda_peak_allocated_bytes": 46192128,
  "macs": "not yet profiled; do not infer from parameter count"
}
Training worker PID: 301
{
  "architecture": "unet",
  "device": "cpu",
  "hardware": "x86_64",
  "threads": 4,
  "batch_size": 1,
  "image_size": 256,
  "parameters": 1965569,
  "p50_ms": 73.3791920006297,
  "p95_ms": 90.85001699986606,
  "warmups": 20,
  "timed_iterations": 200,
  "scope": "model pass only; no image decoding/transfer",
  "recovery_checkpoint_bytes": 23705833,
  "fp32_weight_bytes": 7862276,
  "cuda_peak_allocated_bytes": null,
  "macs": "not yet profile

## Export before ending the session

Run this cell even after an earlier failure. It exports code, manifest, tests,
logs, evaluation, and the latest/best checkpoint snapshots for each model. Other
historical snapshots remain on Kaggle without deletion. Download the ZIP and
checksum from Kaggle Files/Output, and retain a persistent Kaggle output version.
This cell does not itself verify that an external backup was downloaded.


In [5]:
import zipfile,hashlib
backup=Path('/kaggle/working/phase2_stabilization_results.zip')
selected=set()
for path in project.rglob('*'):
    if not path.is_file() or '__pycache__' in path.parts or '.pytest_cache' in path.parts: continue
    if path.suffix=='.pt': continue
    selected.add(path)
for architecture in ['unet','r2au']:
    folder=evidence/architecture/'checkpoints'
    for which in ['latest','best']:
        pointer=folder/(which+'.json')
        if pointer.exists():
            entry=json.loads(pointer.read_text())
            path=folder/entry['file']
            assert path.resolve().parent==folder.resolve()
            assert hashlib.sha256(path.read_bytes()).hexdigest()==entry['sha256']
            selected.add(path)
with zipfile.ZipFile(backup,'w',zipfile.ZIP_DEFLATED) as z:
    for path in sorted(selected): z.write(path,path.relative_to(project).as_posix())
digest=hashlib.sha256(backup.read_bytes()).hexdigest()
backup.with_suffix('.zip.sha256').write_text(digest+'  '+backup.name+'\n')
print('Download:',backup)
print('SHA256:',digest)
print('Historical snapshots remain on Kaggle. Latest and best are included in this ZIP.')


Download: /kaggle/working/phase2_stabilization_results.zip
SHA256: 578ea01fb8389abd0904f63ed621192bc6708b6d76cac8ca9c2840e01d1a5040
Historical snapshots remain on Kaggle. Latest and best are included in this ZIP.
